In [ ]:
""" 
Vinicius Costa Gandolfi         RA: 245274 
"""

!apt-get install -y -qq coinor-cbc #MILP Grande Porte 
!pip install -q pyomo
import numpy as np
import pandas as pd
from pyomo.environ import *

# Caixeiro Viajante

Maximize $\sum_{j=1}^n$ $\sum_{i=1}^n$ $c_{ij, i≠j}$ * $x_{ij, i≠j}$ 

$\sum_{i=1, i≠j}^n$ $x_{ij}$ i = 1, 2, ..., n

$\sum_{j=1, i≠j}^n$ $x_{ij}$ j = 1, 2, ..., n

$u_{i}$ - $u_{j}$ ≤ n-1, 2 ≤ i ≠ j ≤ n

$x_{ij}$ ∈ {0, 1}  i, j = 1, 2, ..., n i ≠ j

$u_{i}$ ∈ $R^+$ i = 1, 2, ..., n


$x_j$ = {0, 1} ∀ j = 1... n

In [ ]:
dist = np.array([[0, 162, 379, 765, 1590, 2430, 3070, 5020],
                [162, 0, 215, 600, 1540, 2260, 2900, 5120],
                [378.5, 215, 0, 392, 1520, 2060, 2690, 5060],
                [765, 600, 392, 0, 1450, 1650, 2300, 4820],
                [1590, 1540, 1520, 1450, 0, 2280, 2770, 3620],
                [2430, 2260, 2060, 1650, 2280, 0, 642, 4420],
                [3070, 2900, 2690, 2300, 2770, 642, 0, 4360],
                [5020, 5120, 5060, 4820, 3620, 4420, 4360, 0]])
n = len(dist)

In [ ]:
# Inicializa o problema p que será resolvido com o solver 
model = ConcreteModel()

In [ ]:
#Model
model = ConcreteModel()

#Indexes for the cities
model.M = RangeSet(n)                
model.N = RangeSet(n)

#Index for the dummy variable u
model.U = RangeSet(2,n)

In [ ]:
model.x = Var(model.N,model.M, within=Binary)

#Dummy variable ui
model.u = Var(model.N, within=NonNegativeIntegers,bounds=(0,n-1))

In [ ]:
expr = sum(dist[i-1, j-1] * model.x[i, j] for i in model.M for j in model.N if i!= j)
model.c = Objective(sense=minimize, expr=expr)

In [ ]:
def rule_const1(model,M):
    return sum(model.x[i,M] for i in model.N if i!=M ) == 1

model.const1 = Constraint(model.M,rule=rule_const1)

In [ ]:
def rule_const2(model,N):
    return sum(model.x[N,j] for j in model.M if j!=N) == 1

model.rest2 = Constraint(model.N,rule=rule_const2)

In [ ]:
def rule_const3(model,i,j):
    if i!=j: 
        return model.u[i] - model.u[j] + model.x[i,j] * n <= n-1
    else:
        #Yeah, this else doesn't say anything
        return model.u[i] - model.u[i] == 0 
    
model.rest3 = Constraint(model.U,model.N,rule=rule_const3)

In [ ]:
model.pprint()

2 Set Declarations
    rest3_index : Size=1, Index=None, Ordered=True
        Key  : Dimen : Domain : Size : Members
        None :     2 :    U*N :   56 : {(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (4, 8), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (5, 8), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (6, 8), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (7, 6), (7, 7), (7, 8), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (8, 6), (8, 7), (8, 8)}
    x_index : Size=1, Index=None, Ordered=True
        Key  : Dimen : Domain : Size : Members
        None :     2 :    N*M :   64 : {(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (4, 8), (5, 1

In [ ]:
solver = SolverFactory('cbc', executable='/usr/bin/cbc')
results = solver.solve(model, tee=True)

Welcome to the CBC MILP Solver 
Version: 2.9.9 
Build Date: Aug 21 2017 

command line - /usr/bin/cbc -printingOptions all -import /tmp/tmpanq09i8m.pyomo.lp -stat=1 -solve -solu /tmp/tmpanq09i8m.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
Presolve 65 (-8) rows, 63 (-2) columns and 252 (-8) elements
Statistics for presolved model
Original problem has 64 integers (56 of which binary)
Presolved problem has 63 integers (56 of which binary)
==== 7 zero objective 30 different
==== absolute objective values 30 different
==== for integers 7 zero objective 30 different
==== for integers absolute objective values 30 different
===== end objective counts


Problem has 65 rows, 63 columns (56 with objective) and 252 elements
Column breakdown:
0 of type 0.0->inf, 7 of type 0.0->up, 0 of type lo->inf, 
0 of type lo->up, 0 of type free, 0 of type fixed, 
0 of type -inf->0.0, 0 of type -inf->up, 56 of type 0.0->1.0 
Row breakdown:
0 of type E 0.0, 16 of type E 

In [ ]:
l = list(model.x.keys())
for i in l:
  if model.x[i]() != 0:
    print(i,'--', model.x[i]())

(1, 1) -- None
(1, 5) -- 1.0
(2, 1) -- 1.0
(2, 2) -- None
(3, 2) -- 1.0
(3, 3) -- None
(4, 3) -- 1.0
(4, 4) -- None
(5, 5) -- None
(5, 8) -- 1.0
(6, 4) -- 1.0
(6, 6) -- None
(7, 6) -- 1.0
(7, 7) -- None
(8, 7) -- 1.0
(8, 8) -- None
